<a href="https://colab.research.google.com/github/smousavi05/Harvard-EPS-210/blob/main/Introduction_to_Keras_TensorFlow_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Introduction to Keras & TensorFlow

### From Tensors to Training Neural Networks

**Course:** EPS-210 — Machine Learning in Earth & Planetary Sciences  
**Harvard University**  
**Estimated Time:** ~60 minutes

---

## What is TensorFlow & Keras?

**TensorFlow** is Google's open-source deep learning framework. **Keras** is its high-level API — a clean, user-friendly interface for building and training neural networks. Since TF 2.0, Keras is the *default* way to use TensorFlow.

<div style="text-align:center; margin: 20px 0;">
<svg width="720" height="200" viewBox="0 0 720 200" xmlns="http://www.w3.org/2000/svg">
  <!-- TF Engine box -->
  <rect x="30" y="90" width="660" height="85" rx="14" fill="#FF6D00" stroke="#E65100" stroke-width="2.5" opacity="0.15"/>
  <text x="360" y="160" text-anchor="middle" font-size="13" fill="#E65100" font-weight="bold">TensorFlow Engine</text>
  <text x="360" y="178" text-anchor="middle" font-size="10" fill="#BF360C">Computation graphs · GPU/TPU acceleration · Automatic differentiation · Deployment</text>

  <!-- Keras layer -->
  <rect x="60" y="30" width="600" height="70" rx="14" fill="#FF6D00" stroke="#E65100" stroke-width="2.5"/>
  <text x="360" y="58" text-anchor="middle" font-size="16" font-weight="bold" fill="white">Keras API</text>
  <text x="360" y="80" text-anchor="middle" font-size="11" fill="#FFE0B2">Layers · Models · Losses · Optimizers · Callbacks · model.fit()</text>

  <!-- Three sub-boxes inside Keras -->
  <rect x="85" y="40" width="130" height="22" rx="5" fill="white" opacity="0.2"/>
  <text x="150" y="55" text-anchor="middle" font-size="9" fill="#FFF3E0" font-weight="bold">Sequential</text>

  <rect x="235" y="40" width="130" height="22" rx="5" fill="white" opacity="0.2"/>
  <text x="300" y="55" text-anchor="middle" font-size="9" fill="#FFF3E0" font-weight="bold">Functional</text>

  <rect x="385" y="40" width="130" height="22" rx="5" fill="white" opacity="0.2"/>
  <text x="450" y="55" text-anchor="middle" font-size="9" fill="#FFF3E0" font-weight="bold">Subclassing</text>

  <!-- Annotation -->
  <text x="580" y="55" text-anchor="middle" font-size="9" fill="#FFF3E0" font-style="italic">← 3 ways to build</text>

  <!-- Label -->
  <text x="360" y="15" text-anchor="middle" font-size="12" font-style="italic" fill="#6D4C41">
    "Keras is designed for human beings, not machines." — François Chollet
  </text>
</svg>
</div>

### Keras vs. PyTorch — Two Philosophies

| | **Keras / TensorFlow** | **PyTorch** |
|---|---|---|
| **Paradigm** | `model.compile()` + `model.fit()` | Manual training loop |
| **Flexibility** | High-level, batteries-included | Low-level, full control |
| **Strengths** | Rapid prototyping, deployment, TPU support | Research, custom architectures |
| **Used by** | Google, industry production, TFLite mobile | Meta, academia, research |

### Roadmap

| Part | Topic | Time |
|------|-------|------|
| 1 | **Tensors** — `tf.Tensor`, shapes, ops, GPU | ~12 min |
| 2 | **GradientTape** — TensorFlow's automatic differentiation | ~10 min |
| 3 | **Building Models** — Sequential, Functional, Subclassing | ~10 min |
| 4 | **Compile & Fit** — The Keras training workflow | ~12 min |
| 5 | **Putting It Together** — Earthquake magnitude regression | ~16 min |

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version:      {keras.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU:                {gpus[0].name}")
else:
    print("Running on CPU (this notebook works fine on CPU)")

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

---

## Part 1: Tensors — The Foundation (~12 min)

A `tf.Tensor` is a multi-dimensional array — the fundamental data structure in TensorFlow. It's conceptually identical to a NumPy array but can run on GPUs/TPUs.

<div style="text-align:center; margin: 20px 0;">
<svg width="720" height="250" viewBox="0 0 720 250" xmlns="http://www.w3.org/2000/svg">
  <!-- Scalar -->
  <rect x="15" y="30" width="70" height="70" rx="8" fill="#FF6D00" stroke="#E65100" stroke-width="2"/>
  <text x="50" y="72" text-anchor="middle" font-size="20" font-weight="bold" fill="white">7</text>
  <text x="50" y="125" text-anchor="middle" font-size="12" font-weight="bold" fill="#E65100">Scalar</text>
  <text x="50" y="142" text-anchor="middle" font-size="10" fill="#868E96">rank-0 tensor</text>
  <text x="50" y="158" text-anchor="middle" font-size="10" fill="#868E96" font-family="monospace">shape: ()</text>
  
  <!-- Vector -->
  <g transform="translate(130, 30)">
    <rect x="0" y="0" width="30" height="70" rx="6" fill="#2979FF" stroke="#1565C0" stroke-width="2"/>
    <text x="15" y="20" text-anchor="middle" font-size="12" fill="white" font-weight="bold">3</text>
    <text x="15" y="40" text-anchor="middle" font-size="12" fill="white" font-weight="bold">1</text>
    <text x="15" y="60" text-anchor="middle" font-size="12" fill="white" font-weight="bold">4</text>
  </g>
  <text x="145" y="125" text-anchor="middle" font-size="12" font-weight="bold" fill="#1565C0">Vector</text>
  <text x="145" y="142" text-anchor="middle" font-size="10" fill="#868E96">rank-1 tensor</text>
  <text x="145" y="158" text-anchor="middle" font-size="10" fill="#868E96" font-family="monospace">shape: (3,)</text>
  
  <!-- Matrix -->
  <g transform="translate(215, 30)">
    <rect x="0" y="0" width="90" height="70" rx="6" fill="#00C853" stroke="#00891B" stroke-width="2"/>
    <text x="18" y="20" text-anchor="middle" font-size="11" fill="white" font-weight="bold">1</text>
    <text x="45" y="20" text-anchor="middle" font-size="11" fill="white" font-weight="bold">2</text>
    <text x="72" y="20" text-anchor="middle" font-size="11" fill="white" font-weight="bold">3</text>
    <text x="18" y="45" text-anchor="middle" font-size="11" fill="white" font-weight="bold">4</text>
    <text x="45" y="45" text-anchor="middle" font-size="11" fill="white" font-weight="bold">5</text>
    <text x="72" y="45" text-anchor="middle" font-size="11" fill="white" font-weight="bold">6</text>
    <text x="18" y="66" text-anchor="middle" font-size="9" fill="#C8E6C9">...</text>
    <text x="45" y="66" text-anchor="middle" font-size="9" fill="#C8E6C9">...</text>
    <text x="72" y="66" text-anchor="middle" font-size="9" fill="#C8E6C9">...</text>
  </g>
  <text x="260" y="125" text-anchor="middle" font-size="12" font-weight="bold" fill="#00891B">Matrix</text>
  <text x="260" y="142" text-anchor="middle" font-size="10" fill="#868E96">rank-2 tensor</text>
  <text x="260" y="158" text-anchor="middle" font-size="10" fill="#868E96" font-family="monospace">shape: (3, 3)</text>
  
  <!-- 3D Tensor -->
  <g transform="translate(360, 18)">
    <rect x="20" y="0" width="80" height="60" rx="4" fill="#AA00FF" stroke="#7200CA" stroke-width="1.5" opacity="0.5"/>
    <rect x="10" y="12" width="80" height="60" rx="4" fill="#AA00FF" stroke="#7200CA" stroke-width="1.5" opacity="0.7"/>
    <rect x="0" y="24" width="80" height="60" rx="4" fill="#AA00FF" stroke="#7200CA" stroke-width="2"/>
    <text x="40" y="50" text-anchor="middle" font-size="10" fill="white" font-weight="bold">Batch</text>
    <text x="40" y="65" text-anchor="middle" font-size="10" fill="white" font-weight="bold">of data</text>
  </g>
  <text x="410" y="125" text-anchor="middle" font-size="12" font-weight="bold" fill="#7200CA">3-D Tensor</text>
  <text x="410" y="142" text-anchor="middle" font-size="10" fill="#868E96">e.g. time series</text>
  <text x="410" y="158" text-anchor="middle" font-size="10" fill="#868E96" font-family="monospace">shape: (B, T, F)</text>
  
  <!-- 4D Tensor -->
  <g transform="translate(510, 10)">
    <rect x="25" y="0" width="60" height="50" rx="3" fill="#FFD600" stroke="#F9A825" stroke-width="1" opacity="0.35"/>
    <rect x="15" y="10" width="60" height="50" rx="3" fill="#FFD600" stroke="#F9A825" stroke-width="1" opacity="0.5"/>
    <rect x="5" y="20" width="60" height="50" rx="3" fill="#FFD600" stroke="#F9A825" stroke-width="1" opacity="0.7"/>
    <rect x="0" y="28" width="60" height="50" rx="3" fill="#FFD600" stroke="#F9A825" stroke-width="2"/>
    <line x1="0" y1="44" x2="60" y2="44" stroke="#F9A825" stroke-width="0.5" opacity="0.4"/>
    <line x1="0" y1="60" x2="60" y2="60" stroke="#F9A825" stroke-width="0.5" opacity="0.4"/>
    <line x1="20" y1="28" x2="20" y2="78" stroke="#F9A825" stroke-width="0.5" opacity="0.4"/>
    <line x1="40" y1="28" x2="40" y2="78" stroke="#F9A825" stroke-width="0.5" opacity="0.4"/>
  </g>
  <text x="545" y="125" text-anchor="middle" font-size="12" font-weight="bold" fill="#F9A825">4-D Tensor</text>
  <text x="545" y="142" text-anchor="middle" font-size="10" fill="#868E96">e.g. images</text>
  <text x="545" y="158" text-anchor="middle" font-size="10" fill="#868E96" font-family="monospace">shape: (B, H, W, C)</text>

  <!-- n-D Tensor -->
  <g transform="translate(635, 25)">
    <circle cx="35" cy="35" r="32" fill="none" stroke="#868E96" stroke-width="2" stroke-dasharray="5,3"/>
    <text x="35" y="32" text-anchor="middle" font-size="18" font-weight="bold" fill="#495057">n-D</text>
    <text x="35" y="48" text-anchor="middle" font-size="10" fill="#868E96">...</text>
  </g>
  <text x="670" y="125" text-anchor="middle" font-size="12" font-weight="bold" fill="#495057">n-D Tensor</text>
  <text x="670" y="142" text-anchor="middle" font-size="10" fill="#868E96">any shape</text>
  <text x="670" y="158" text-anchor="middle" font-size="10" fill="#868E96" font-family="monospace">shape: (...)</text>

  <!-- Bottom annotations -->
  <text x="50" y="185" text-anchor="middle" font-size="9" fill="#ADB5BD">loss value</text>
  <text x="145" y="185" text-anchor="middle" font-size="9" fill="#ADB5BD">features</text>
  <text x="260" y="185" text-anchor="middle" font-size="9" fill="#ADB5BD">data table</text>
  <text x="410" y="185" text-anchor="middle" font-size="9" fill="#ADB5BD">seismograms</text>
  <text x="545" y="185" text-anchor="middle" font-size="9" fill="#ADB5BD">satellite tiles</text>

  <!-- Note about channel order -->
  <rect x="100" y="210" width="520" height="28" rx="6" fill="#FFF3E0" stroke="#FF6D00" stroke-width="1.5"/>
  <text x="360" y="228" text-anchor="middle" font-size="10" fill="#E65100" font-weight="bold">
    ⚠️ TensorFlow uses channels-LAST by default: (B, H, W, C).  PyTorch uses channels-FIRST: (B, C, H, W).
  </text>
</svg>
</div>

### 1.1 Creating Tensors

In [ ]:
# === Creating tensors ===

# From Python lists
t1 = tf.constant([1, 2, 3])
print(f"From list:    {t1.numpy()}   shape={t1.shape}   dtype={t1.dtype}")

# From NumPy
arr = np.array([[1.0, 2.0], [3.0, 4.0]])
t2 = tf.constant(arr)
print(f"From NumPy:   shape={t2.shape}   dtype={t2.dtype}")

# Common constructors
t_zeros = tf.zeros([3, 4])            # 3×4 matrix of zeros
t_ones  = tf.ones([2, 3, 5])          # 2×3×5 tensor of ones
t_rand  = tf.random.normal([4, 3])    # Normal(0,1) random
t_range = tf.range(0, 10, 0.5)        # Like np.arange
t_eye   = tf.eye(3)                   # 3×3 identity

print(f"\nzeros:  {t_zeros.shape}")
print(f"ones:   {t_ones.shape}")
print(f"normal: {t_rand.shape}")
print(f"range:  {t_range.shape}  → {t_range[:5].numpy()}...")
print(f"eye:\n{t_eye.numpy()}")

In [ ]:
# === Key difference from NumPy: tf.Tensor is IMMUTABLE ===
# You cannot do t[0] = 5 with a tf.Tensor.
# For mutable tensors, use tf.Variable (used for model weights).

t_const = tf.constant([1.0, 2.0, 3.0])
t_var   = tf.Variable([1.0, 2.0, 3.0])

print(f"tf.constant: {type(t_const).__name__}  — immutable")
print(f"tf.Variable: {type(t_var).__name__}  — mutable (used for weights)")

# Variables can be modified in-place
t_var[0].assign(99.0)
print(f"\nAfter assign: {t_var.numpy()}")

# Cast between types
t_int = tf.constant([1, 2, 3])
t_float = tf.cast(t_int, tf.float32)
print(f"\nCast: {t_int.dtype} → {t_float.dtype}")

### 1.2 Tensor Operations

In [ ]:
a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
b = tf.constant([[5.0, 6.0], [7.0, 8.0]])

# Element-wise operations
print("a + b =", (a + b).numpy().tolist())
print("a * b =", (a * b).numpy().tolist())          # Element-wise multiply
print("a @ b =", (a @ b).numpy().tolist())           # Matrix multiply
print("tf.transpose(a) =", tf.transpose(a).numpy().tolist())

# Reductions
print(f"\nsum:  {tf.reduce_sum(a).numpy():.1f}")
print(f"mean: {tf.reduce_mean(a).numpy():.1f}")
print(f"max:  {tf.reduce_max(a).numpy():.1f}")
print(f"sum along rows (axis=1): {tf.reduce_sum(a, axis=1).numpy().tolist()}")
print(f"sum along cols (axis=0): {tf.reduce_sum(a, axis=0).numpy().tolist()}")

In [ ]:
# === Reshaping ===
t = tf.range(12)  # [0, 1, 2, ..., 11]
print(f"Original: {t.shape} → {t.numpy().tolist()}")

# Reshape to 2D
t_2d = tf.reshape(t, [3, 4])
print(f"\nreshape([3,4]):\n{t_2d.numpy()}")

# Reshape to 3D
t_3d = tf.reshape(t, [2, 2, 3])
print(f"\nreshape([2,2,3]):\n{t_3d.numpy()}")

# -1 infers the dimension
t_auto = tf.reshape(t, [4, -1])
print(f"\nreshape([4,-1]): {t_auto.shape}")

# expand_dims / squeeze
t_flat = tf.random.normal([5])
print(f"\nOriginal:         {t_flat.shape}")
print(f"expand_dims(0):   {tf.expand_dims(t_flat, 0).shape}  ← add batch dim")
print(f"expand_dims(-1):  {tf.expand_dims(t_flat, -1).shape}  ← add feature dim")

In [ ]:
# === Indexing & Slicing ===
t = tf.cast(tf.reshape(tf.range(20), [4, 5]), tf.float32)
print(f"t:\n{t.numpy()}\n")

print(f"t[0]       = {t[0].numpy().tolist()}       ← first row")
print(f"t[:, 0]    = {t[:, 0].numpy().tolist()}    ← first column")
print(f"t[1:3, 2:] = \n{t[1:3, 2:].numpy()}  ← slice")

# Boolean masking
mask = t > 10
print(f"\nt[t > 10]  = {tf.boolean_mask(t, mask).numpy().tolist()}")

### 1.3 GPU Acceleration

TensorFlow automatically places tensors on the GPU when available. You rarely need to manage device placement manually.

<div style="text-align:center; margin: 15px 0;">
<svg width="600" height="130" viewBox="0 0 600 130" xmlns="http://www.w3.org/2000/svg">
  <!-- CPU -->
  <rect x="20" y="20" width="160" height="85" rx="10" fill="#E8EAF6" stroke="#3949AB" stroke-width="2"/>
  <text x="100" y="48" text-anchor="middle" font-size="14" font-weight="bold" fill="#283593">CPU</text>
  <text x="100" y="68" text-anchor="middle" font-size="10" fill="#5C6BC0">tf.device('/CPU:0')</text>
  <text x="100" y="88" text-anchor="middle" font-size="10" fill="#7986CB">data loading, preprocessing</text>
  
  <!-- Arrow -->
  <line x1="190" y1="55" x2="265" y2="55" stroke="#FF6D00" stroke-width="3" marker-end="url(#arrTF)"/>
  <text x="228" y="48" text-anchor="middle" font-size="9" fill="#E65100" font-weight="bold">automatic</text>
  
  <!-- GPU -->
  <rect x="275" y="20" width="160" height="85" rx="10" fill="#FF6D00" stroke="#E65100" stroke-width="2"/>
  <text x="355" y="48" text-anchor="middle" font-size="14" font-weight="bold" fill="white">GPU / TPU</text>
  <text x="355" y="68" text-anchor="middle" font-size="10" fill="#FFE0B2">tf.device('/GPU:0')</text>
  <text x="355" y="88" text-anchor="middle" font-size="10" fill="#FFE0B2">matrix ops, training</text>

  <!-- Note -->
  <rect x="460" y="28" width="125" height="65" rx="8" fill="#E8F5E9" stroke="#43A047" stroke-width="1.5"/>
  <text x="522" y="50" text-anchor="middle" font-size="10" fill="#2E7D32" font-weight="bold">TF handles</text>
  <text x="522" y="65" text-anchor="middle" font-size="10" fill="#2E7D32" font-weight="bold">placement</text>
  <text x="522" y="80" text-anchor="middle" font-size="10" fill="#4CAF50">automatically!</text>

  <defs>
    <marker id="arrTF" markerWidth="8" markerHeight="6" refX="8" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#FF6D00"/></marker>
  </defs>
</svg>
</div>

In [ ]:
# TensorFlow automatically uses GPU when available
t = tf.random.normal([1000, 1000])
print(f"Tensor device: {t.device}")

# Explicit device placement (rarely needed)
with tf.device('/CPU:0'):
    t_cpu = tf.random.normal([100, 100])
    print(f"Forced CPU: {t_cpu.device}")

if gpus:
    with tf.device('/GPU:0'):
        t_gpu = tf.random.normal([100, 100])
        print(f"Forced GPU: {t_gpu.device}")

### 🏋️ Exercise 1

Create a tensor `seismogram` of shape `(32, 1000, 3)` representing a batch of 32 three-component seismograms with 1000 time samples each. Fill it with random normal values.

In [ ]:
# YOUR CODE HERE
seismogram = tf.random.normal([32, 1000, 3])

print(f"Shape:    {seismogram.shape}")
print(f"Rank:     {tf.rank(seismogram).numpy()}")
print(f"Elements: {tf.size(seismogram).numpy():,}")
print(f"Memory:   {tf.size(seismogram).numpy() * 4 / 1024:.1f} KB (float32 = 4 bytes)")

---

## Part 2: GradientTape — Automatic Differentiation (~10 min)

TensorFlow uses **`tf.GradientTape`** — a context manager that "records" operations on tensors so it can compute gradients afterward via backpropagation.

<div style="text-align:center; margin: 20px 0;">
<svg width="680" height="260" viewBox="0 0 680 260" xmlns="http://www.w3.org/2000/svg">
  <!-- Title -->
  <text x="340" y="22" text-anchor="middle" font-size="14" font-weight="bold" fill="#495057">GradientTape records:  L = (w·x + b - y)²</text>

  <!-- Tape border -->
  <rect x="10" y="32" width="660" height="180" rx="16" fill="none" stroke="#FF6D00" stroke-width="2" stroke-dasharray="8,4"/>
  <text x="45" y="50" font-size="10" font-family="monospace" fill="#E65100" font-weight="bold">with tf.GradientTape() as tape:</text>

  <!-- w node -->
  <circle cx="80" cy="100" r="26" fill="#FF6D00" stroke="#E65100" stroke-width="2"/>
  <text x="80" y="96" text-anchor="middle" font-size="12" font-weight="bold" fill="white">w</text>
  <text x="80" y="110" text-anchor="middle" font-size="8" fill="#FFE0B2">Variable</text>

  <!-- x node -->
  <circle cx="80" cy="175" r="26" fill="#E8EAF6" stroke="#5C6BC0" stroke-width="2"/>
  <text x="80" y="179" text-anchor="middle" font-size="12" font-weight="bold" fill="#283593">x</text>

  <!-- Multiply -->
  <circle cx="195" cy="132" r="22" fill="#2979FF" stroke="#1565C0" stroke-width="2"/>
  <text x="195" y="137" text-anchor="middle" font-size="16" font-weight="bold" fill="white">×</text>
  <line x1="104" y1="108" x2="175" y2="127" stroke="#78909C" stroke-width="1.8"/>
  <line x1="104" y1="168" x2="175" y2="140" stroke="#78909C" stroke-width="1.8"/>

  <!-- b node -->
  <circle cx="195" cy="195" r="26" fill="#FF6D00" stroke="#E65100" stroke-width="2"/>
  <text x="195" y="191" text-anchor="middle" font-size="12" font-weight="bold" fill="white">b</text>
  <text x="195" y="205" text-anchor="middle" font-size="8" fill="#FFE0B2">Variable</text>

  <!-- Add -->
  <circle cx="310" cy="155" r="22" fill="#2979FF" stroke="#1565C0" stroke-width="2"/>
  <text x="310" y="160" text-anchor="middle" font-size="16" font-weight="bold" fill="white">+</text>
  <line x1="217" y1="137" x2="290" y2="150" stroke="#78909C" stroke-width="1.8"/>
  <line x1="217" y1="188" x2="290" y2="162" stroke="#78909C" stroke-width="1.8"/>

  <!-- y node -->
  <circle cx="310" cy="80" r="26" fill="#E8EAF6" stroke="#5C6BC0" stroke-width="2"/>
  <text x="310" y="84" text-anchor="middle" font-size="12" font-weight="bold" fill="#283593">y</text>

  <!-- Subtract -->
  <circle cx="420" cy="115" r="22" fill="#2979FF" stroke="#1565C0" stroke-width="2"/>
  <text x="420" y="121" text-anchor="middle" font-size="16" font-weight="bold" fill="white">−</text>
  <line x1="332" y1="147" x2="400" y2="122" stroke="#78909C" stroke-width="1.8"/>
  <line x1="332" y1="90" x2="400" y2="110" stroke="#78909C" stroke-width="1.8"/>

  <!-- Square -->
  <circle cx="520" cy="115" r="22" fill="#2979FF" stroke="#1565C0" stroke-width="2"/>
  <text x="520" y="120" text-anchor="middle" font-size="12" font-weight="bold" fill="white">( )²</text>
  <line x1="442" y1="115" x2="498" y2="115" stroke="#78909C" stroke-width="1.8"/>

  <!-- Loss -->
  <rect x="570" y="90" width="55" height="50" rx="10" fill="#00C853" stroke="#00891B" stroke-width="2.5"/>
  <text x="597" y="120" text-anchor="middle" font-size="15" font-weight="bold" fill="white">L</text>
  <line x1="542" y1="115" x2="570" y2="115" stroke="#78909C" stroke-width="1.8"/>

  <!-- Gradient call -->
  <text x="340" y="240" text-anchor="middle" font-size="11" fill="#E65100" font-weight="bold" font-family="monospace">
    grads = tape.gradient(L, [w, b])  →  returns [∂L/∂w, ∂L/∂b]
  </text>

  <!-- Legend -->
  <circle cx="100" cy="248" r="7" fill="#FF6D00" stroke="#E65100" stroke-width="1.5"/>
  <text x="114" y="252" font-size="9" fill="#495057">= tf.Variable (watched)</text>
  <circle cx="290" cy="248" r="7" fill="#E8EAF6" stroke="#5C6BC0" stroke-width="1.5"/>
  <text x="304" y="252" font-size="9" fill="#495057">= tf.constant (data)</text>
  <circle cx="450" cy="248" r="7" fill="#2979FF" stroke="#1565C0" stroke-width="1.5"/>
  <text x="464" y="252" font-size="9" fill="#495057">= operation</text>
</svg>
</div>

In [ ]:
# === GradientTape in action ===

# Learnable parameters (tf.Variable — tape watches these automatically)
w = tf.Variable(2.0)
b = tf.Variable(1.0)

# Data
x = tf.constant(3.0)
y = tf.constant(10.0)

# Record operations inside the tape
with tf.GradientTape() as tape:
    pred = w * x + b           # ŷ = wx + b
    loss = (pred - y) ** 2     # L = (ŷ − y)²

# Compute gradients
grads = tape.gradient(loss, [w, b])

print(f"Prediction: w*x + b = {w.numpy():.1f}*{x.numpy():.1f} + {b.numpy():.1f} = {pred.numpy():.1f}")
print(f"Target:     y = {y.numpy():.1f}")
print(f"Loss:       (ŷ − y)² = ({pred.numpy():.1f} − {y.numpy():.1f})² = {loss.numpy():.1f}")
print(f"\n∂L/∂w = {grads[0].numpy():.1f}")
print(f"∂L/∂b = {grads[1].numpy():.1f}")

# Manual check:
print(f"\n✓ ∂L/∂w = 2·(wx+b−y)·x = 2·({pred.numpy():.0f}−{y.numpy():.0f})·{x.numpy():.0f} = {2*(pred.numpy()-y.numpy())*x.numpy():.0f}")

In [ ]:
# === Gradient descent by hand ===
# Fit y = 3x + 1

w = tf.Variable(0.0)
b = tf.Variable(0.0)
lr = 0.01

# Data
tf.random.set_seed(42)
X = tf.linspace(-2.0, 2.0, 50)
Y = 3.0 * X + 1.0 + tf.random.normal([50], stddev=0.3)

losses = []
w_history, b_history = [], []

for step in range(200):
    with tf.GradientTape() as tape:
        pred = w * X + b
        loss = tf.reduce_mean((pred - Y) ** 2)

    grads = tape.gradient(loss, [w, b])

    # Update
    w.assign_sub(lr * grads[0])   # w -= lr * ∂L/∂w
    b.assign_sub(lr * grads[1])   # b -= lr * ∂L/∂b

    losses.append(loss.numpy())
    w_history.append(w.numpy())
    b_history.append(b.numpy())

print(f"After 200 steps:")
print(f"  w = {w.numpy():.4f}  (true: 3.0)")
print(f"  b = {b.numpy():.4f}  (true: 1.0)")
print(f"  loss = {losses[-1]:.6f}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].semilogy(losses, color='#E65100', linewidth=1.5)
axes[0].set_xlabel('Step', fontsize=11)
axes[0].set_ylabel('MSE Loss', fontsize=11)
axes[0].set_title('Training Loss', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(w_history, b_history, '-o', color='#1565C0', markersize=2, lw=0.8, alpha=0.7)
axes[1].plot(w_history[0], b_history[0], 'o', color='red', ms=10, label='Start', zorder=5)
axes[1].plot(w_history[-1], b_history[-1], '*', color='gold', ms=15,
             markeredgecolor='black', label='End', zorder=5)
axes[1].plot(3.0, 1.0, 'x', color='green', ms=12, markeredgewidth=3, label='True', zorder=5)
axes[1].set_xlabel('w', fontsize=11); axes[1].set_ylabel('b', fontsize=11)
axes[1].set_title('Parameter Space Trajectory', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].scatter(X.numpy(), Y.numpy(), s=15, alpha=0.6, color='steelblue', label='Data')
x_plot = np.linspace(-2.5, 2.5, 100)
axes[2].plot(x_plot, w.numpy()*x_plot + b.numpy(), 'r-', lw=2.5,
             label=f'Fit: y={w.numpy():.2f}x+{b.numpy():.2f}')
axes[2].plot(x_plot, 3*x_plot+1, 'g--', lw=1.5, label='True: y=3x+1', alpha=0.7)
axes[2].set_xlabel('x', fontsize=11); axes[2].set_ylabel('y', fontsize=11)
axes[2].set_title('Linear Regression Result', fontsize=12, fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key takeaway:** `tf.GradientTape` records operations, `tape.gradient()` computes derivatives. But in Keras, you almost never write this manually — `model.fit()` handles it all.

---

## Part 3: Building Models — Three Ways (~10 min)

Keras offers three ways to build models, from simplest to most flexible:

<div style="text-align:center; margin: 20px 0;">
<svg width="700" height="320" viewBox="0 0 700 320" xmlns="http://www.w3.org/2000/svg">
  <!-- Sequential -->
  <rect x="15" y="10" width="210" height="300" rx="14" fill="#FFF3E0" stroke="#FF6D00" stroke-width="2.5"/>
  <text x="120" y="38" text-anchor="middle" font-size="14" font-weight="bold" fill="#E65100">① Sequential</text>
  <text x="120" y="57" text-anchor="middle" font-size="10" fill="#BF360C" font-style="italic">Stack of layers, linear flow</text>
  
  <!-- Layer blocks -->
  <rect x="35" y="72" width="170" height="28" rx="6" fill="#FF6D00" stroke="#E65100" stroke-width="1.5"/>
  <text x="120" y="90" text-anchor="middle" font-size="10" fill="white" font-weight="bold">Dense(64, 'relu')</text>
  <line x1="120" y1="100" x2="120" y2="108" stroke="#BF360C" stroke-width="2" marker-end="url(#aK)"/>
  <rect x="35" y="112" width="170" height="28" rx="6" fill="#FF6D00" stroke="#E65100" stroke-width="1.5"/>
  <text x="120" y="130" text-anchor="middle" font-size="10" fill="white" font-weight="bold">Dense(32, 'relu')</text>
  <line x1="120" y1="140" x2="120" y2="148" stroke="#BF360C" stroke-width="2" marker-end="url(#aK)"/>
  <rect x="35" y="152" width="170" height="28" rx="6" fill="#FF6D00" stroke="#E65100" stroke-width="1.5"/>
  <text x="120" y="170" text-anchor="middle" font-size="10" fill="white" font-weight="bold">Dense(1)</text>

  <text x="120" y="205" text-anchor="middle" font-size="9" fill="#4E342E" font-family="monospace">model = Sequential([</text>
  <text x="120" y="218" text-anchor="middle" font-size="9" fill="#4E342E" font-family="monospace">  Dense(64, 'relu'),</text>
  <text x="120" y="231" text-anchor="middle" font-size="9" fill="#4E342E" font-family="monospace">  Dense(32, 'relu'),</text>
  <text x="120" y="244" text-anchor="middle" font-size="9" fill="#4E342E" font-family="monospace">  Dense(1)</text>
  <text x="120" y="257" text-anchor="middle" font-size="9" fill="#4E342E" font-family="monospace">])</text>

  <rect x="40" y="270" width="160" height="24" rx="5" fill="#C8E6C9" stroke="#43A047" stroke-width="1"/>
  <text x="120" y="286" text-anchor="middle" font-size="9" fill="#2E7D32" font-weight="bold">✓ Best for: simple pipelines</text>

  <!-- Functional -->
  <rect x="245" y="10" width="210" height="300" rx="14" fill="#E3F2FD" stroke="#2979FF" stroke-width="2.5"/>
  <text x="350" y="38" text-anchor="middle" font-size="14" font-weight="bold" fill="#1565C0">② Functional</text>
  <text x="350" y="57" text-anchor="middle" font-size="10" fill="#0D47A1" font-style="italic">DAG of layers, multi-I/O</text>
  
  <!-- Branching diagram -->
  <rect x="310" y="72" width="80" height="24" rx="5" fill="#2979FF" stroke="#1565C0" stroke-width="1.5"/>
  <text x="350" y="88" text-anchor="middle" font-size="9" fill="white" font-weight="bold">Input</text>
  
  <!-- Two branches -->
  <line x1="330" y1="96" x2="300" y2="110" stroke="#1565C0" stroke-width="1.5"/>
  <line x1="370" y1="96" x2="400" y2="110" stroke="#1565C0" stroke-width="1.5"/>
  
  <rect x="270" y="112" width="60" height="22" rx="5" fill="#2979FF" stroke="#1565C0" stroke-width="1"/>
  <text x="300" y="127" text-anchor="middle" font-size="8" fill="white" font-weight="bold">Dense</text>
  <rect x="370" y="112" width="60" height="22" rx="5" fill="#2979FF" stroke="#1565C0" stroke-width="1"/>
  <text x="400" y="127" text-anchor="middle" font-size="8" fill="white" font-weight="bold">Conv1D</text>
  
  <!-- Merge -->
  <line x1="300" y1="134" x2="350" y2="150" stroke="#1565C0" stroke-width="1.5"/>
  <line x1="400" y1="134" x2="350" y2="150" stroke="#1565C0" stroke-width="1.5"/>
  
  <rect x="315" y="150" width="70" height="22" rx="5" fill="#2979FF" stroke="#1565C0" stroke-width="1.5"/>
  <text x="350" y="165" text-anchor="middle" font-size="8" fill="white" font-weight="bold">Concatenate</text>
  <line x1="350" y1="172" x2="350" y2="180" stroke="#1565C0" stroke-width="1.5" marker-end="url(#aK)"/>
  <rect x="315" y="182" width="70" height="22" rx="5" fill="#2979FF" stroke="#1565C0" stroke-width="1.5"/>
  <text x="350" y="197" text-anchor="middle" font-size="8" fill="white" font-weight="bold">Output</text>

  <text x="350" y="220" text-anchor="middle" font-size="9" fill="#0D47A1" font-family="monospace">inp = Input((8,))</text>
  <text x="350" y="233" text-anchor="middle" font-size="9" fill="#0D47A1" font-family="monospace">x = Dense(64)(inp)</text>
  <text x="350" y="246" text-anchor="middle" font-size="9" fill="#0D47A1" font-family="monospace">out = Dense(1)(x)</text>
  <text x="350" y="259" text-anchor="middle" font-size="9" fill="#0D47A1" font-family="monospace">Model(inp, out)</text>

  <rect x="270" y="270" width="160" height="24" rx="5" fill="#C8E6C9" stroke="#43A047" stroke-width="1"/>
  <text x="350" y="286" text-anchor="middle" font-size="9" fill="#2E7D32" font-weight="bold">✓ Best for: multi-input/output</text>

  <!-- Subclassing -->
  <rect x="475" y="10" width="210" height="300" rx="14" fill="#F3E5F5" stroke="#AA00FF" stroke-width="2.5"/>
  <text x="580" y="38" text-anchor="middle" font-size="14" font-weight="bold" fill="#7200CA">③ Subclassing</text>
  <text x="580" y="57" text-anchor="middle" font-size="10" fill="#4A148C" font-style="italic">Full Python, PyTorch-like</text>

  <text x="580" y="82" text-anchor="middle" font-size="9" fill="#4A148C" font-family="monospace">class MyModel(Model):</text>
  <text x="580" y="100" text-anchor="middle" font-size="9" fill="#4A148C" font-family="monospace">  def __init__(self):</text>
  <text x="580" y="118" text-anchor="middle" font-size="9" fill="#4A148C" font-family="monospace">    self.d1 = Dense(64)</text>
  <text x="580" y="136" text-anchor="middle" font-size="9" fill="#4A148C" font-family="monospace">    self.d2 = Dense(1)</text>
  <text x="580" y="158" text-anchor="middle" font-size="9" fill="#4A148C" font-family="monospace">  def call(self, x):</text>
  <text x="580" y="176" text-anchor="middle" font-size="9" fill="#4A148C" font-family="monospace">    x = tf.nn.relu(</text>
  <text x="580" y="194" text-anchor="middle" font-size="9" fill="#4A148C" font-family="monospace">      self.d1(x))</text>
  <text x="580" y="212" text-anchor="middle" font-size="9" fill="#4A148C" font-family="monospace">    return self.d2(x)</text>

  <rect x="500" y="235" width="160" height="44" rx="5" fill="#FCE4EC" stroke="#C62828" stroke-width="1"/>
  <text x="580" y="252" text-anchor="middle" font-size="8" fill="#B71C1C">⚠️ Most flexible but loses</text>
  <text x="580" y="264" text-anchor="middle" font-size="8" fill="#B71C1C">model.summary() & serialization</text>

  <rect x="500" y="270" width="160" height="24" rx="5" fill="#C8E6C9" stroke="#43A047" stroke-width="1"/>
  <text x="580" y="286" text-anchor="middle" font-size="9" fill="#2E7D32" font-weight="bold">✓ Best for: research, custom logic</text>

  <defs>
    <marker id="aK" markerWidth="6" markerHeight="5" refX="3" refY="5" orient="auto">
      <path d="M0,0 L3,5 L6,0" fill="none" stroke="inherit" stroke-width="1.5"/>
    </marker>
  </defs>
</svg>
</div>

In [ ]:
# === Method 1: Sequential ===
# The simplest — just a list of layers.

model_seq = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(4,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),
])

model_seq.summary()

In [ ]:
# === Method 2: Functional API ===
# More flexible — can handle multiple inputs/outputs, shared layers, branches.

inputs = layers.Input(shape=(4,), name='features')
x = layers.Dense(64, activation='relu', name='hidden1')(inputs)
x = layers.Dense(32, activation='relu', name='hidden2')(x)
outputs = layers.Dense(1, name='output')(x)

model_func = keras.Model(inputs=inputs, outputs=outputs, name='functional_net')
model_func.summary()

In [ ]:
# === Method 3: Subclassing (PyTorch-like) ===

class SimpleNet(keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = layers.Dense(64, activation='relu')
        self.dense2 = layers.Dense(32, activation='relu')
        self.out    = layers.Dense(1)

    def call(self, x):
        x = self.dense1(x)
        x = self.dense2(x)
        return self.out(x)

model_sub = SimpleNet()
# Must call once to build weights
_ = model_sub(tf.zeros([1, 4]))
model_sub.summary()

In [ ]:
# === Common layers reference ===

layers_ref = {
    'Dense(units, activation)':     'Fully connected: y = activation(Wx + b)',
    'Conv2D(filters, kernel)':      '2D convolution (images)',
    'LSTM(units)':                  'Long Short-Term Memory (sequences)',
    'BatchNormalization()':         'Batch normalization',
    'Dropout(rate)':               'Randomly zero elements (regularization)',
    'Flatten()':                    'Reshape to 1D (after Conv layers)',
    'Concatenate()':               'Merge multiple inputs',
    'Input(shape)':                'Define model input (Functional API)',
}

print(f"{'Keras Layer':<32} {'Description'}")
print("=" * 72)
for layer, desc in layers_ref.items():
    print(f"{layer:<32} {desc}")

---

## Part 4: Compile & Fit — The Keras Way (~12 min)

Keras replaces the manual training loop with **`model.compile()`** + **`model.fit()`**. This is the single biggest difference from PyTorch.

<div style="text-align:center; margin: 20px 0;">
<svg width="680" height="290" viewBox="0 0 680 290" xmlns="http://www.w3.org/2000/svg">
  <!-- Step 1: Build -->
  <rect x="30" y="20" width="190" height="105" rx="14" fill="#2979FF" stroke="#1565C0" stroke-width="2.5"/>
  <text x="125" y="48" text-anchor="middle" font-size="13" font-weight="bold" fill="white">① BUILD</text>
  <text x="50" y="70" font-size="10" fill="#BBDEFB" font-family="monospace">model = Sequential([</text>
  <text x="50" y="85" font-size="10" fill="#BBDEFB" font-family="monospace">  Dense(64, 'relu'),</text>
  <text x="50" y="100" font-size="10" fill="#BBDEFB" font-family="monospace">  Dense(1)</text>
  <text x="50" y="115" font-size="10" fill="#BBDEFB" font-family="monospace">])</text>

  <!-- Arrow -->
  <line x1="225" y1="72" x2="255" y2="72" stroke="#37474F" stroke-width="2.5" marker-end="url(#aKM)"/>

  <!-- Step 2: Compile -->
  <rect x="260" y="20" width="190" height="105" rx="14" fill="#FF6D00" stroke="#E65100" stroke-width="2.5"/>
  <text x="355" y="48" text-anchor="middle" font-size="13" font-weight="bold" fill="white">② COMPILE</text>
  <text x="280" y="70" font-size="10" fill="#FFE0B2" font-family="monospace">model.compile(</text>
  <text x="280" y="85" font-size="10" fill="#FFE0B2" font-family="monospace">  optimizer='adam',</text>
  <text x="280" y="100" font-size="10" fill="#FFE0B2" font-family="monospace">  loss='mse',</text>
  <text x="280" y="115" font-size="10" fill="#FFE0B2" font-family="monospace">  metrics=['mae'])</text>

  <!-- Arrow -->
  <line x1="455" y1="72" x2="485" y2="72" stroke="#37474F" stroke-width="2.5" marker-end="url(#aKM)"/>

  <!-- Step 3: Fit -->
  <rect x="490" y="20" width="170" height="105" rx="14" fill="#00C853" stroke="#00891B" stroke-width="2.5"/>
  <text x="575" y="48" text-anchor="middle" font-size="13" font-weight="bold" fill="white">③ FIT</text>
  <text x="510" y="70" font-size="10" fill="#C8E6C9" font-family="monospace">history = model.fit(</text>
  <text x="510" y="85" font-size="10" fill="#C8E6C9" font-family="monospace">  X_train, y_train,</text>
  <text x="510" y="100" font-size="10" fill="#C8E6C9" font-family="monospace">  epochs=100,</text>
  <text x="510" y="115" font-size="10" fill="#C8E6C9" font-family="monospace">  validation_split=.2)</text>

  <!-- What compile does -->
  <rect x="30" y="150" width="300" height="120" rx="12" fill="#FFF3E0" stroke="#FF6D00" stroke-width="1.5"/>
  <text x="180" y="172" text-anchor="middle" font-size="11" font-weight="bold" fill="#E65100">What compile() configures:</text>
  <text x="45" y="195" font-size="10" fill="#4E342E">🎯 <tspan font-weight="bold">Loss function</tspan> — what to minimize (mse, crossentropy)</text>
  <text x="45" y="215" font-size="10" fill="#4E342E">⚙️ <tspan font-weight="bold">Optimizer</tspan> — how to update weights (adam, sgd, rmsprop)</text>
  <text x="45" y="235" font-size="10" fill="#4E342E">📊 <tspan font-weight="bold">Metrics</tspan> — what to monitor (accuracy, mae, r²)</text>
  <text x="45" y="255" font-size="9" fill="#795548" font-style="italic">→ Sets up the forward/backward/update pipeline automatically</text>

  <!-- What fit does -->
  <rect x="350" y="150" width="310" height="120" rx="12" fill="#E8F5E9" stroke="#43A047" stroke-width="1.5"/>
  <text x="505" y="172" text-anchor="middle" font-size="11" font-weight="bold" fill="#2E7D32">What fit() handles:</text>
  <text x="365" y="195" font-size="10" fill="#1B5E20">🔄 Training loop (all epochs + batches)</text>
  <text x="365" y="215" font-size="10" fill="#1B5E20">📈 Loss/metric computation each batch</text>
  <text x="365" y="235" font-size="10" fill="#1B5E20">✅ Validation after each epoch</text>
  <text x="365" y="255" font-size="10" fill="#1B5E20">📝 Returns history object with curves</text>

  <defs>
    <marker id="aKM" markerWidth="8" markerHeight="6" refX="8" refY="3" orient="auto">
      <path d="M0,0 L8,3 L0,6 Z" fill="#37474F"/>
    </marker>
  </defs>
</svg>
</div>

In [ ]:
# === Compile & Fit: the Keras way ===

# Synthetic data: y = sin(x₁) + x₂² − x₃ + 0.5·x₄ + noise
np.random.seed(42)
N = 500
X_data = np.random.randn(N, 4).astype(np.float32)
y_data = (np.sin(X_data[:, 0]) + X_data[:, 1]**2
          - X_data[:, 2] + 0.5 * X_data[:, 3]
          + 0.1 * np.random.randn(N)).astype(np.float32)

print(f"Data: X={X_data.shape}, y={y_data.shape}")

In [ ]:
# Build
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(4,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(1),
])

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='mse',
    metrics=['mae'],
)

# Fit!
history = model.fit(
    X_data, y_data,
    epochs=100,
    batch_size=32,
    validation_split=0.2,   # Use 20% as validation
    verbose=0,              # Quiet during training
)

print(f"\n✅ Training complete!")
print(f"Final train loss: {history.history['loss'][-1]:.4f}")
print(f"Final val   loss: {history.history['val_loss'][-1]:.4f}")

In [ ]:
# The history object contains everything
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(history.history['loss'], label='Train', color='#1565C0', lw=1.5)
axes[0].semilogy(history.history['val_loss'], label='Validation', color='#C62828', lw=1.5, ls='--')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('MSE Loss (log)', fontsize=11)
axes[0].set_title('Loss Curves', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train', color='#1565C0', lw=1.5)
axes[1].plot(history.history['val_mae'], label='Validation', color='#C62828', lw=1.5, ls='--')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Mean Absolute Error', fontsize=11)
axes[1].set_title('MAE Curves', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# === Evaluate & Predict ===

# Evaluate on new data
X_test_data = np.random.randn(100, 4).astype(np.float32)
y_test_data = (np.sin(X_test_data[:, 0]) + X_test_data[:, 1]**2
               - X_test_data[:, 2] + 0.5 * X_test_data[:, 3]
               + 0.1 * np.random.randn(100)).astype(np.float32)

test_loss, test_mae = model.evaluate(X_test_data, y_test_data, verbose=0)
print(f"Test MSE:  {test_loss:.4f}")
print(f"Test MAE:  {test_mae:.4f}")

# Predict
predictions = model.predict(X_test_data[:5], verbose=0)
print(f"\nPredictions: {predictions.flatten().tolist()[:5]}")
print(f"Actual:      {y_test_data[:5].tolist()}")

### Key Keras Functions

| Function | Purpose |
|----------|--------|
| `model.compile(optimizer, loss, metrics)` | Configure the training pipeline |
| `model.fit(X, y, epochs, batch_size, ...)` | Train the model |
| `model.evaluate(X, y)` | Compute loss & metrics on test data |
| `model.predict(X)` | Forward pass (no training) |
| `model.summary()` | Print architecture and param count |
| `model.save('path')` | Save entire model (weights + arch) |
| `keras.models.load_model('path')` | Load a saved model |

---

## Part 5: Putting It Together — Earthquake Magnitude Regression (~16 min)

Let's apply everything to a realistic Earth science problem: **predicting earthquake magnitude from waveform features**.

<div style="text-align:center; margin: 20px 0;">
<svg width="680" height="150" viewBox="0 0 680 150" xmlns="http://www.w3.org/2000/svg">
  <!-- Input -->
  <rect x="10" y="20" width="150" height="110" rx="10" fill="#E3F2FD" stroke="#2979FF" stroke-width="2"/>
  <text x="85" y="42" text-anchor="middle" font-size="11" font-weight="bold" fill="#1565C0">Input Features (8)</text>
  <text x="85" y="60" text-anchor="middle" font-size="9" fill="#37474F">log₁₀(P amplitude)</text>
  <text x="85" y="73" text-anchor="middle" font-size="9" fill="#37474F">S−P time (s)</text>
  <text x="85" y="86" text-anchor="middle" font-size="9" fill="#37474F">duration (s)</text>
  <text x="85" y="99" text-anchor="middle" font-size="9" fill="#37474F">peak frequency (Hz)</text>
  <text x="85" y="112" text-anchor="middle" font-size="9" fill="#78909C">+ 4 more ...</text>

  <line x1="165" y1="75" x2="195" y2="75" stroke="#37474F" stroke-width="2" marker-end="url(#aEQ)"/>

  <!-- Network -->
  <rect x="200" y="15" width="270" height="120" rx="12" fill="#FFF3E0" stroke="#FF6D00" stroke-width="2"/>
  <text x="335" y="38" text-anchor="middle" font-size="11" font-weight="bold" fill="#E65100">Keras Sequential</text>
  <text x="335" y="58" text-anchor="middle" font-size="10" font-family="monospace" fill="#37474F">Dense(64, 'relu') + Dropout(0.2)</text>
  <text x="335" y="73" text-anchor="middle" font-size="10" font-family="monospace" fill="#37474F">Dense(32, 'relu') + Dropout(0.2)</text>
  <text x="335" y="88" text-anchor="middle" font-size="10" font-family="monospace" fill="#37474F">Dense(16, 'relu')</text>
  <text x="335" y="103" text-anchor="middle" font-size="10" font-family="monospace" fill="#37474F">Dense(1)  ← predicted M̂</text>
  <text x="335" y="125" text-anchor="middle" font-size="9" fill="#BF360C" font-style="italic">compile(optimizer='adam', loss='mse')</text>

  <line x1="475" y1="75" x2="505" y2="75" stroke="#37474F" stroke-width="2" marker-end="url(#aEQ)"/>

  <!-- Output -->
  <rect x="510" y="30" width="155" height="90" rx="10" fill="#E8F5E9" stroke="#43A047" stroke-width="2"/>
  <text x="587" y="58" text-anchor="middle" font-size="11" font-weight="bold" fill="#2E7D32">Output</text>
  <text x="587" y="78" text-anchor="middle" font-size="13" fill="#1B5E20">Predicted M̂</text>
  <text x="587" y="100" text-anchor="middle" font-size="9" fill="#78909C">fit() → evaluate()</text>

  <defs>
    <marker id="aEQ" markerWidth="8" markerHeight="6" refX="8" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#37474F"/></marker>
  </defs>
</svg>
</div>

In [ ]:
# ============================================================
# Generate realistic synthetic earthquake data
# ============================================================
np.random.seed(42)
N_SAMPLES = 3000

magnitudes = np.clip(np.random.exponential(1.0, N_SAMPLES) + 1.0, 1.0, 8.0)
distances = 10 ** np.random.uniform(0.5, 2.5, N_SAMPLES)

features = np.column_stack([
    magnitudes * 0.8 - np.log10(distances) * 1.2 + np.random.normal(0, 0.3, N_SAMPLES),
    distances / 8.0 + np.random.normal(0, 0.5, N_SAMPLES),
    10 ** (0.4 * magnitudes - 0.5) + np.random.normal(0, 2, N_SAMPLES),
    10 ** (1.5 - 0.2 * magnitudes) + np.random.normal(0, 1, N_SAMPLES),
    3.0 - 0.3 * magnitudes + np.random.normal(0, 0.4, N_SAMPLES),
    magnitudes * 0.9 - np.log10(distances) * 1.1 + np.random.normal(0, 0.25, N_SAMPLES),
    0.5 + np.random.normal(0, 0.15, N_SAMPLES),
    0.01 * magnitudes + 0.02 + np.random.normal(0, 0.005, N_SAMPLES),
]).astype(np.float32)

feature_names = ['log₁₀(P-amp)', 'S−P time', 'Duration', 'Peak freq',
                 'Spectral ratio', 'log₁₀(S-amp)', 'P/S ratio', 'Coda decay']

# Normalize
feat_mean = features.mean(axis=0)
feat_std  = features.std(axis=0)
features  = (features - feat_mean) / (feat_std + 1e-8)

# Split
idx = np.random.permutation(N_SAMPLES)
n_train = int(0.8 * N_SAMPLES)
X_train, y_train = features[idx[:n_train]], magnitudes[idx[:n_train]]
X_test, y_test   = features[idx[n_train:]], magnitudes[idx[n_train:]]

print(f"Features: {feature_names}")
print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

In [ ]:
# Visualize
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, (ax, name) in enumerate(zip(axes.flatten(), feature_names)):
    # Unnormalize for display
    x_orig = X_train[:, i] * feat_std[i] + feat_mean[i]
    sc = ax.scatter(x_orig, y_train, s=3, alpha=0.3,
                    c=y_train, cmap='YlOrRd', vmin=1, vmax=6)
    ax.set_xlabel(name, fontsize=10)
    ax.set_ylabel('Magnitude' if i % 4 == 0 else '', fontsize=10)
    ax.grid(True, alpha=0.2)

plt.suptitle('Seismological Features vs. Earthquake Magnitude',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Build the model with Callbacks
# ============================================================

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(8,)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1),
], name='MagnitudeNet')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae'],
)

model.summary()

# Callbacks: early stopping + learning rate reduction
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, verbose=1
    ),
]

print("\n🚀 Training with EarlyStopping + ReduceLROnPlateau...")

In [ ]:
# Train
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=64,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=0,
)

n_epochs_trained = len(history.history['loss'])
print(f"\n✅ Training stopped after {n_epochs_trained} epochs")
print(f"Best val loss: {min(history.history['val_loss']):.4f}")
print(f"Best val MAE:  {min(history.history['val_mae']):.4f}")

In [ ]:
# Evaluate on test set
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test MSE:  {test_loss:.4f}")
print(f"Test RMSE: {np.sqrt(test_loss):.4f} magnitude units")
print(f"Test MAE:  {test_mae:.4f} magnitude units")

In [ ]:
# ============================================================
# Final evaluation and visualization
# ============================================================
y_pred = model.predict(X_test, verbose=0).flatten()
residuals = y_pred - y_test
rmse = np.sqrt(np.mean(residuals**2))
mae  = np.mean(np.abs(residuals))
r2   = 1 - np.sum(residuals**2) / np.sum((y_test - y_test.mean())**2)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. Loss curves
axes[0, 0].semilogy(history.history['loss'], label='Train', color='#1565C0', lw=1.5)
axes[0, 0].semilogy(history.history['val_loss'], label='Validation', color='#C62828', lw=1.5, ls='--')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('MSE Loss')
axes[0, 0].set_title('Training Progress', fontweight='bold')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

# 2. Predicted vs Actual
sc = axes[0, 1].scatter(y_test, y_pred, s=12, alpha=0.5,
                        c=np.abs(residuals), cmap='RdYlGn_r', vmin=0, vmax=1)
axes[0, 1].plot([1, 8], [1, 8], 'k--', lw=1.5)
axes[0, 1].set_xlabel('True Magnitude'); axes[0, 1].set_ylabel('Predicted Magnitude')
axes[0, 1].set_title(f'Predicted vs. True  (R² = {r2:.3f})', fontweight='bold')
axes[0, 1].set_aspect('equal')
plt.colorbar(sc, ax=axes[0, 1], label='|Residual|', shrink=0.8)
axes[0, 1].grid(True, alpha=0.3)

# 3. Residual histogram
axes[1, 0].hist(residuals, bins=50, color='steelblue', edgecolor='white', density=True)
axes[1, 0].axvline(0, color='red', ls='--', lw=1.5)
axes[1, 0].set_xlabel('Residual (Pred − True)'); axes[1, 0].set_ylabel('Density')
axes[1, 0].set_title(f'Residuals  (RMSE={rmse:.3f}, MAE={mae:.3f})', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# 4. Error vs magnitude
axes[1, 1].scatter(y_test, np.abs(residuals), s=12, alpha=0.4, color='coral')
mag_bins = np.arange(1, 7.5, 0.5)
bin_means = [np.mean(np.abs(residuals[(y_test >= m) & (y_test < m+0.5)]))
             for m in mag_bins if np.sum((y_test >= m) & (y_test < m+0.5)) > 5]
valid_bins = [m+0.25 for m in mag_bins
              if np.sum((y_test >= m) & (y_test < m+0.5)) > 5]
axes[1, 1].plot(valid_bins, bin_means[:len(valid_bins)], 'k-o', lw=2, ms=6,
                label='Bin mean', zorder=5)
axes[1, 1].set_xlabel('True Magnitude'); axes[1, 1].set_ylabel('|Prediction Error|')
axes[1, 1].set_title('Error vs. Magnitude', fontweight='bold')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Earthquake Magnitude Prediction — Results',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Save and load
# ============================================================

# Save entire model (architecture + weights + optimizer state)
model.save('magnitude_model.keras')
print(f"Model saved ({os.path.getsize('magnitude_model.keras')/1024:.1f} KB)")

# Load
model_loaded = keras.models.load_model('magnitude_model.keras')

# Verify identical predictions
pred_orig = model.predict(X_test[:3], verbose=0).flatten()
pred_load = model_loaded.predict(X_test[:3], verbose=0).flatten()
print(f"\nOriginal: {pred_orig.tolist()}")
print(f"Loaded:   {pred_load.tolist()}")
print("✅ Identical!")

---

## 📝 Summary & Cheat Sheet

<div style="text-align:center; margin: 20px 0;">
<svg width="700" height="380" viewBox="0 0 700 380" xmlns="http://www.w3.org/2000/svg">
  <text x="350" y="24" text-anchor="middle" font-size="16" font-weight="bold" fill="#212529">Keras / TensorFlow at a Glance</text>

  <!-- Column 1: Data -->
  <rect x="15" y="40" width="210" height="325" rx="12" fill="#E3F2FD" stroke="#2979FF" stroke-width="2"/>
  <text x="120" y="65" text-anchor="middle" font-size="13" font-weight="bold" fill="#1565C0">📦 Tensors</text>
  <text x="25" y="90" font-size="10" font-family="monospace" fill="#37474F">tf.constant([...])</text>
  <text x="25" y="108" font-size="10" font-family="monospace" fill="#37474F">tf.Variable([...])</text>
  <text x="25" y="126" font-size="10" font-family="monospace" fill="#37474F">tf.random.normal([B,T,F])</text>
  <text x="25" y="148" font-size="10" font-family="monospace" fill="#78909C"># Reshape</text>
  <text x="25" y="166" font-size="10" font-family="monospace" fill="#37474F">tf.reshape(t, [3,-1])</text>
  <text x="25" y="184" font-size="10" font-family="monospace" fill="#37474F">tf.expand_dims(t, 0)</text>
  <text x="25" y="206" font-size="10" font-family="monospace" fill="#78909C"># Ops</text>
  <text x="25" y="224" font-size="10" font-family="monospace" fill="#37474F">tf.reduce_mean(t)</text>
  <text x="25" y="242" font-size="10" font-family="monospace" fill="#37474F">tf.matmul(a, b)  # a@b</text>
  <text x="25" y="264" font-size="10" font-family="monospace" fill="#78909C"># Gradients</text>
  <text x="25" y="282" font-size="10" font-family="monospace" fill="#37474F">with GradientTape():</text>
  <text x="25" y="300" font-size="10" font-family="monospace" fill="#37474F">  loss = f(x)</text>
  <text x="25" y="318" font-size="10" font-family="monospace" fill="#37474F">tape.gradient(</text>
  <text x="25" y="336" font-size="10" font-family="monospace" fill="#37474F">  loss, vars)</text>

  <!-- Column 2: Model -->
  <rect x="245" y="40" width="210" height="325" rx="12" fill="#FFF3E0" stroke="#FF6D00" stroke-width="2"/>
  <text x="350" y="65" text-anchor="middle" font-size="13" font-weight="bold" fill="#E65100">🧠 Model</text>
  <text x="255" y="90" font-size="10" font-family="monospace" fill="#37474F">model = Sequential([</text>
  <text x="255" y="108" font-size="10" font-family="monospace" fill="#37474F">  Dense(64, 'relu'),</text>
  <text x="255" y="126" font-size="10" font-family="monospace" fill="#37474F">  Dropout(0.2),</text>
  <text x="255" y="144" font-size="10" font-family="monospace" fill="#37474F">  Dense(1)</text>
  <text x="255" y="162" font-size="10" font-family="monospace" fill="#37474F">])</text>
  <text x="255" y="188" font-size="10" font-family="monospace" fill="#78909C"># Key layers</text>
  <text x="255" y="206" font-size="10" font-family="monospace" fill="#37474F">Dense(units, act)</text>
  <text x="255" y="224" font-size="10" font-family="monospace" fill="#37474F">Conv2D(filters, k)</text>
  <text x="255" y="242" font-size="10" font-family="monospace" fill="#37474F">LSTM(units)</text>
  <text x="255" y="260" font-size="10" font-family="monospace" fill="#37474F">Dropout(rate)</text>
  <text x="255" y="278" font-size="10" font-family="monospace" fill="#37474F">BatchNormalization()</text>
  <text x="255" y="300" font-size="10" font-family="monospace" fill="#78909C"># Save / Load</text>
  <text x="255" y="318" font-size="10" font-family="monospace" fill="#37474F">model.save('m.keras')</text>
  <text x="255" y="336" font-size="10" font-family="monospace" fill="#37474F">load_model('m.keras')</text>

  <!-- Column 3: Training -->
  <rect x="475" y="40" width="210" height="325" rx="12" fill="#E8F5E9" stroke="#43A047" stroke-width="2"/>
  <text x="580" y="65" text-anchor="middle" font-size="13" font-weight="bold" fill="#2E7D32">🏋️ Train & Evaluate</text>
  <text x="485" y="90" font-size="10" font-family="monospace" fill="#78909C"># Compile</text>
  <text x="485" y="108" font-size="10" font-family="monospace" fill="#37474F">model.compile(</text>
  <text x="485" y="126" font-size="10" font-family="monospace" fill="#E65100">  optimizer='adam',</text>
  <text x="485" y="144" font-size="10" font-family="monospace" fill="#C62828">  loss='mse',</text>
  <text x="485" y="162" font-size="10" font-family="monospace" fill="#1565C0">  metrics=['mae'])</text>
  <text x="485" y="188" font-size="10" font-family="monospace" fill="#78909C"># Train</text>
  <text x="485" y="206" font-size="10" font-family="monospace" fill="#37474F">hist = model.fit(</text>
  <text x="485" y="224" font-size="10" font-family="monospace" fill="#37474F">  X, y, epochs=100,</text>
  <text x="485" y="242" font-size="10" font-family="monospace" fill="#37474F">  validation_split=.2,</text>
  <text x="485" y="260" font-size="10" font-family="monospace" fill="#37474F">  callbacks=[...])</text>
  <text x="485" y="286" font-size="10" font-family="monospace" fill="#78909C"># Evaluate</text>
  <text x="485" y="304" font-size="10" font-family="monospace" fill="#37474F">model.evaluate(X, y)</text>
  <text x="485" y="322" font-size="10" font-family="monospace" fill="#37474F">model.predict(X_new)</text>
  <text x="485" y="340" font-size="10" font-family="monospace" fill="#37474F">model.summary()</text>
</svg>
</div>

### Keras vs. PyTorch — Side-by-Side

| Task | **Keras** | **PyTorch** |
|------|-----------|-------------|
| Create tensor | `tf.constant([1,2,3])` | `torch.tensor([1,2,3])` |
| Random | `tf.random.normal([3,4])` | `torch.randn(3,4)` |
| Reshape | `tf.reshape(t, [3,-1])` | `t.reshape(3,-1)` |
| Matmul | `tf.matmul(a,b)` or `a @ b` | `torch.matmul(a,b)` or `a @ b` |
| Gradients | `GradientTape` context mgr | `loss.backward()` |
| Define model | `Sequential([...])` | `class Net(nn.Module)` |
| Train | `model.fit(X, y, ...)` | Manual loop: forward/backward/step |
| Evaluate | `model.evaluate(X, y)` | Manual loop: `with torch.no_grad()` |
| Save | `model.save('m.keras')` | `torch.save(model.state_dict(), ...)` |
| Channel order | Channels-**last** `(B,H,W,C)` | Channels-**first** `(B,C,H,W)` |

### Discussion Questions

1. **Ease vs. control:** Keras `model.fit()` hides the training loop. When is this an advantage? When might you prefer PyTorch's explicit loop?

2. **Callbacks:** We used `EarlyStopping` and `ReduceLROnPlateau`. In PyTorch, you'd implement these manually. What other callbacks might be useful for Earth science problems? (Hint: `ModelCheckpoint`, `TensorBoard`, custom loggers.)

3. **`tf.Tensor` is immutable; `tf.Variable` is mutable.** Why does TensorFlow make this distinction? How does it relate to the computation graph?

4. **Error vs. magnitude:** Large earthquakes have higher prediction error. Beyond training data imbalance, what physical reasons might cause this?

5. **From features to waveforms:** Modern seismology models (PhaseNet, EQTransformer) process raw waveforms using `Conv1D` layers. How would you modify this Keras model to accept 3-component seismograms of length 1000?

### Going Further

- **CNNs for images:** Use `Conv2D` + `MaxPool2D` for satellite imagery or crater detection
- **LSTMs for time series:** Use `LSTM` layers for streamflow forecasting or seismic signal processing
- **Transfer learning:** Load pretrained models with `keras.applications` and fine-tune
- **TensorBoard:** Add `keras.callbacks.TensorBoard` for interactive training visualization
- **TFLite:** Export models for mobile/edge deployment with `tf.lite.TFLiteConverter`
- **TPU training:** Google Colab offers free TPU access — Keras supports it natively

### References

- [Keras Documentation](https://keras.io/)
- [TensorFlow Documentation](https://www.tensorflow.org/)
- Chollet, F. (2021). Deep Learning with Python, 2nd Edition. Manning.
- Abadi, M., et al. (2016). TensorFlow: A System for Large-Scale Machine Learning. OSDI.